# PixArt-α — External Heads Concept Demo

Companion notebook for *"Responsible Text-to-Image Diffusion: Interpretable and Linearly Controllable Semantics for Fair and Safe Generation"* (ICML 2026).

This notebook downloads four pretrained concept vectors (cartoon, Van Gogh, male, female) from the Hugging Face Hub repo [`Mattsh21/responsible-t2i-diffusion`](https://huggingface.co/Mattsh21/responsible-t2i-diffusion), loads PixArt-α once, and produces a **baseline vs concept** side-by-side comparison for each concept.

**Make sure you're on a GPU runtime:** *Runtime → Change runtime type → T4 / A100 / L4*.

## 0. Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors sentencepiece huggingface_hub

## 1. Imports

In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from diffusers import PixArtAlphaPipeline, DPMSolverMultistepScheduler
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), "No GPU detected. Switch to a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Shared configuration

These settings apply to every concept section below. The defaults match the values from the paper (`TARGET_LAYERS = [11..27]`, `TARGET_HEADS = [10, 12, 14]`).

`COEFFICIENTS` is per-concept — change any single value to dial that concept up or down without affecting the others.

In [ ]:
HF_REPO_ID          = "Mattsh21/responsible-t2i-diffusion"
MODEL_ID            = "PixArt-alpha/PixArt-XL-2-1024-MS"
TARGET_LAYERS       = list(range(11, 28))      # layers 11 to 27 inclusive
TARGET_HEADS        = [10, 11, 12, 14]
SEED                = 42
NUM_INFERENCE_STEPS = 20
GUIDANCE_SCALE      = 4.5
DEVICE              = "cuda"
WEIGHT_DTYPE        = torch.float16            # fp16 for Colab GPU memory

# Per-concept coefficients. Tune per concept without touching the rest.
COEFFICIENTS = {
    "cartoon":  30.0,
    "van_gogh": 20.0,
    "male":     90.0,
    "female":   70.0,
}

# Maps each concept to its filename on the Hugging Face Hub.
CONCEPT_FILES = {
    "cartoon":  "external_concept_cartoon.pt",
    "van_gogh": "external_concept_van_gogh.pt",
    "male":     "external_concept_male.pt",
    "female":   "external_concept_female.pt",
}

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

## 3. External heads container + custom attention processor

Lifted from the paper's reference implementation: a container that holds the trained per-head vectors with per-key sequence-length checks, and a delegating attention processor that adds a bias-free projection of those vectors to PixArt's cross-attention output.

In [ ]:
class LoadedExternalHeads:
    """Container for loaded external heads with per-key sequence length support."""

    def __init__(self, state_dict, target_layers, num_heads=16, head_dim=72):
        self.target_layers = target_layers
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.external_heads = {}

        for key, value in state_dict.items():
            nk = key
            if nk.startswith("external_heads."):
                nk = nk[len("external_heads."):]
            self.external_heads[nk] = value

        print(f"  Loaded external heads: layers={target_layers}, "
              f"heads/layer={num_heads}, head_dim={head_dim}, "
              f"tensors={len(self.external_heads)}")

    def get_external_head(self, layer_idx, head_idx, seq_len, device, dtype, target_heads=None):
        if target_heads is not None and head_idx not in target_heads:
            return torch.zeros(seq_len, self.head_dim, device=device, dtype=dtype)
        if layer_idx not in self.target_layers:
            return torch.zeros(seq_len, self.head_dim, device=device, dtype=dtype)

        key = f"layer_{layer_idx}_head_{head_idx}"
        if key not in self.external_heads:
            return torch.zeros(seq_len, self.head_dim, device=device, dtype=dtype)

        full_head = self.external_heads[key].to(device=device, dtype=dtype)
        S_l = full_head.shape[0]
        if S_l != seq_len:
            raise ValueError(
                f"Sequence length mismatch for {key}: stored={S_l}, runtime={seq_len}."
            )
        return full_head

    def get_all_heads_for_layer(self, layer_idx, seq_len, device, dtype, target_heads=None):
        heads = [
            self.get_external_head(layer_idx, h, seq_len, device, dtype, target_heads)
            for h in range(self.num_heads)
        ]
        return torch.stack(heads, dim=0)


class TrueDelegationExternalHeadsProcessor:
    """Calls the original processor, then adds bias-free projected external head residual."""

    def __init__(self, original_processor, layer_idx, attn_module,
                 external_heads_module, coefficient, target_heads=None):
        self.original_processor = original_processor
        self.layer_idx = layer_idx
        self.attn_module = attn_module
        self.external_heads_module = external_heads_module
        self.coefficient = coefficient
        self.target_heads = target_heads
        self.first_call = True

        self.num_heads = getattr(attn_module, "heads", None)
        if self.num_heads is None:
            raise ValueError(f"Layer {layer_idx}: attn module missing 'heads'")

        if hasattr(attn_module.to_q, "out_features"):
            inner_dim = attn_module.to_q.out_features
        elif hasattr(attn_module.to_q, "weight"):
            inner_dim = attn_module.to_q.weight.shape[0]
        else:
            raise ValueError(f"Layer {layer_idx}: cannot determine inner_dim")
        self.head_dim = inner_dim // self.num_heads

        if self.external_heads_module.head_dim != self.head_dim:
            raise ValueError(
                f"Layer {layer_idx}: head_dim mismatch "
                f"({self.external_heads_module.head_dim} vs {self.head_dim})"
            )

    def _apply_to_out_bias_free(self, attn, delta_concat):
        if not hasattr(attn, "to_out") or attn.to_out is None:
            return delta_concat
        to_out = attn.to_out
        _delta_proj = delta_concat
        if isinstance(to_out, torch.nn.ModuleList):
            for _m in to_out:
                if isinstance(_m, torch.nn.Linear):
                    _delta_proj = F.linear(_delta_proj, _m.weight, bias=None)
                else:
                    _delta_proj = _m(_delta_proj)
        elif isinstance(to_out, torch.nn.Sequential):
            _first = to_out[0]
            if isinstance(_first, torch.nn.Linear):
                _delta_proj = F.linear(delta_concat, _first.weight, bias=None)
                for _m in list(to_out)[1:]:
                    _delta_proj = _m(_delta_proj)
            else:
                _delta_proj = to_out(delta_concat)
        elif isinstance(to_out, torch.nn.Linear):
            _delta_proj = F.linear(delta_concat, to_out.weight, bias=None)
        else:
            _delta_proj = to_out(delta_concat)
        return _delta_proj

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, **kwargs):
        orig_out = self.original_processor(
            attn, hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask, **kwargs,
        )
        B, N, _ = hidden_states.shape
        H, d_h = self.num_heads, self.head_dim
        device, dtype = hidden_states.device, hidden_states.dtype

        if self.first_call:
            self.first_call = False

        external_heads = self.external_heads_module.get_all_heads_for_layer(
            self.layer_idx, N, device, dtype, target_heads=self.target_heads
        )
        external_heads = external_heads.unsqueeze(0).expand(B, -1, -1, -1)
        delta_concat = external_heads.transpose(1, 2).reshape(B, N, H * d_h)

        delta_proj = self._apply_to_out_bias_free(attn, delta_concat)
        return orig_out + self.coefficient * delta_proj


def load_external_heads_from_path(checkpoint_path, target_layers):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    print(f"  Loading: {checkpoint_path}")
    raw = torch.load(checkpoint_path, map_location="cpu")
    normalized = {}
    for k, v in raw.items():
        nk = k[len("external_heads."):] if k.startswith("external_heads.") else k
        normalized[nk] = v
    return LoadedExternalHeads(normalized, target_layers, num_heads=16, head_dim=72)


def save_baseline_processors(pipe, target_layers):
    return {
        i: pipe.transformer.transformer_blocks[i].attn2.get_processor()
        for i in target_layers
    }


def setup_external_heads_processors(pipe, target_layers, baseline_processors,
                                    external_heads_module, coefficient, target_heads=None):
    for layer_idx in target_layers:
        cross_attn = pipe.transformer.transformer_blocks[layer_idx].attn2
        custom = TrueDelegationExternalHeadsProcessor(
            original_processor=baseline_processors[layer_idx],
            layer_idx=layer_idx,
            attn_module=cross_attn,
            external_heads_module=external_heads_module,
            coefficient=coefficient,
            target_heads=target_heads,
        )
        cross_attn.set_processor(custom)


def reset_to_baseline_processors(pipe, target_layers, baseline_processors):
    for layer_idx in target_layers:
        pipe.transformer.transformer_blocks[layer_idx].attn2.set_processor(
            baseline_processors[layer_idx]
        )


def generate_image(pipe, prompt, seed, num_inference_steps, guidance_scale):
    gen = torch.Generator(device=pipe.device).manual_seed(seed)
    return pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        generator=gen,
        guidance_scale=guidance_scale,
    ).images[0]


def show_pair(baseline_img, concept_img, concept_name, coefficient, seed, prompt):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(baseline_img); axes[0].axis("off")
    axes[0].set_title(f"baseline (seed {seed})")
    axes[1].imshow(concept_img);  axes[1].axis("off")
    axes[1].set_title(f"+ {concept_name} concept (coef={coefficient})")
    fig.suptitle(prompt, fontsize=10, wrap=True)
    plt.tight_layout()
    plt.show()


def download_concept(concept_name):
    """Download a single concept .pt from the HF Hub repo to CKPT_DIR."""
    fname = CONCEPT_FILES[concept_name]
    local_path = os.path.join(CKPT_DIR, fname)
    if os.path.exists(local_path):
        print(f"  [cached] {local_path}")
        return local_path
    print(f"  Downloading {fname} from {HF_REPO_ID} (~315 MB)...")
    path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=fname,
        local_dir=CKPT_DIR,
    )
    return path


def run_concept_section(pipe, baseline_processors, concept_name, prompt, coefficient):
    """Download checkpoint, generate baseline + concept pair, display side-by-side."""
    # Always start from a clean processor state.
    reset_to_baseline_processors(pipe, TARGET_LAYERS, baseline_processors)

    print(f"\n[1/4] Fetching checkpoint for '{concept_name}'...")
    ckpt_path = download_concept(concept_name)

    print(f"[2/4] Baseline generation (seed={SEED})...")
    baseline_img = generate_image(pipe, prompt, SEED, NUM_INFERENCE_STEPS, GUIDANCE_SCALE)

    print(f"[3/4] Loading external heads...")
    external_heads = load_external_heads_from_path(ckpt_path, TARGET_LAYERS)

    print(f"[4/4] Concept generation (coef={coefficient})...")
    setup_external_heads_processors(
        pipe, TARGET_LAYERS, baseline_processors,
        external_heads, coefficient, target_heads=TARGET_HEADS,
    )
    try:
        concept_img = generate_image(pipe, prompt, SEED, NUM_INFERENCE_STEPS, GUIDANCE_SCALE)
    finally:
        reset_to_baseline_processors(pipe, TARGET_LAYERS, baseline_processors)

    show_pair(baseline_img, concept_img, concept_name, coefficient, SEED, prompt)
    return baseline_img, concept_img

## 4. Load PixArt-α (once)

Loading PixArt-α takes a couple of minutes the first time — the four concept sections below all reuse the same pipeline.

In [ ]:
print("Loading PixArt-α...")
pipe = PixArtAlphaPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=WEIGHT_DTYPE,
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(DEVICE)
print("Pipeline ready.")

# Snapshot original cross-attention processors so every section can revert.
baseline_processors = save_baseline_processors(pipe, TARGET_LAYERS)
print(f"Saved baseline processors for {len(baseline_processors)} layers "
      f"({TARGET_LAYERS[0]}..{TARGET_LAYERS[-1]}).")

## 5. Concept: **Cartoon**

The first run of this cell downloads `cartoon` (~315 MB) into `/content/checkpoints/`. Subsequent runs reuse the cached file.

The strength of injection is `COEFFICIENTS['cartoon']` — change that entry in the config cell to dial this concept up or down.

**Prompt:** *A detailed illustration of a panda in a zoo setting, surrounded by bamboo, rocks, and soft greenery, with bold outlines, cel-shaded surfaces, expressive eyes, slightly exaggerated proportions.*

In [ ]:
prompt_cartoon = (
    'A detailed illustration of a panda in a zoo setting, surrounded by bamboo, rocks, and soft greenery, with bold outlines, cel-shaded surfaces, expressive eyes, slightly exaggerated proportions.'
)

baseline_cartoon, concept_cartoon = run_concept_section(
    pipe                = pipe,
    baseline_processors = baseline_processors,
    concept_name        = 'cartoon',
    prompt              = prompt_cartoon,
    coefficient         = COEFFICIENTS['cartoon'],
)

## 6. Concept: **Van Gogh**

The first run of this cell downloads `van_gogh` (~315 MB) into `/content/checkpoints/`. Subsequent runs reuse the cached file.

The strength of injection is `COEFFICIENTS['van_gogh']` — change that entry in the config cell to dial this concept up or down.

**Prompt:** *people walking along a riverside path with colorful trees*

In [ ]:
prompt_van_gogh = (
    'people walking along a riverside path with colorful trees'
)

baseline_van_gogh, concept_van_gogh = run_concept_section(
    pipe                = pipe,
    baseline_processors = baseline_processors,
    concept_name        = 'van_gogh',
    prompt              = prompt_van_gogh,
    coefficient         = COEFFICIENTS['van_gogh'],
)

## 7. Concept: **Male**

The first run of this cell downloads `male` (~315 MB) into `/content/checkpoints/`. Subsequent runs reuse the cached file.

The strength of injection is `COEFFICIENTS['male']` — change that entry in the config cell to dial this concept up or down.

**Prompt:** *a photo of a nurse in the hospital*

In [ ]:
prompt_male = (
    'a photo of a nurse in the hospital'
)

baseline_male, concept_male = run_concept_section(
    pipe                = pipe,
    baseline_processors = baseline_processors,
    concept_name        = 'male',
    prompt              = prompt_male,
    coefficient         = COEFFICIENTS['male'],
)

## 8. Concept: **Female**

The first run of this cell downloads `female` (~315 MB) into `/content/checkpoints/`. Subsequent runs reuse the cached file.

The strength of injection is `COEFFICIENTS['female']` — change that entry in the config cell to dial this concept up or down.

**Prompt:** *a photo of a doctor in the hospital*

In [ ]:
prompt_female = (
    'a photo of a doctor in the hospital'
)

baseline_female, concept_female = run_concept_section(
    pipe                = pipe,
    baseline_processors = baseline_processors,
    concept_name        = 'female',
    prompt              = prompt_female,
    coefficient         = COEFFICIENTS['female'],
)